In [2]:
import numpy as np
from math import erf

In [ ]:
class Add_layer:

    def __init__(self,inputs,neurons):
        self.weights = np.random.randn(inputs,neurons) * np.sqrt(2/(inputs))
        self.bias = np.zeros((1,neurons))

    def forward(self,inputs):
        self.inputs = inputs
        self.outputs = np.dot(self.inputs,self.weights) + self.bias

    def backward(self,dvalues):

        self.dweights = np.dot(self.inputs.T,dvalues)
        self.dbias = np.sum(dvalues,axis=0,keepdims=True)
        self.dinputs = np.dot(dvalues,self.weights.T)

class Activation_ReLu:

    def forward(self,inputs):
        self.inputs = inputs
        self.outputs = np.maximum(0,inputs)

    def backward(self,dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs<=0]=0

class Activation_Leaky_ReLu:
    def __init__(self,negative_slope=0.01):
        self.negative_slope = negative_slope
    def forward(self,inputs):
        self.inputs = inputs
        self.outputs = (np.maximum(0,inputs) + self.negative_slope * np.minimum(0,inputs))

    def backward(self,dvalues):

        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs<=0] *= self.negative_slope

class Activation_Softmax:

    def forward(self,inputs):
        exp_value = np.exp(inputs - np.max(inputs,axis=1,keepdims=True))
        probabilities = exp_value / np.sum(exp_value,axis=1,keepdims=True)
        self.outputs = probabilities

    def backward(self,dvalues):
        self.dinputs = np.empty_like(dvalues)
        
        for index,(single_output,single_dvalues) in enumerate(zip(self.outputs,dvalues)):

            single_output = single_output.reshape(-1,1)
            jacobian_matrix = (np.diagflat(single_output) - np.dot(single_output,single_output.T))
            self.dinputs[index] = (jacobian_matrix @ single_dvalues)


class Activation_GeLu:

    def forward(self,inputs):
        self.inputs = inputs

        phi_cdf = 0.5 * (1+np.vectorize(erf)(self.inputs/np.sqrt(2)))
        self.outputs = self.inputs * phi_cdf

    def backward(self,dvalues):

        phi_cdf = 0.5 * (1+np.vectorize(erf)(self.inputs/np.sqrt(2)))
        phi_pdf = (1 / np.sqrt(2*np.pi))*np.exp(-(self.inputs**2)/2)

        gelu_derivative = phi_cdf + self.inputs * phi_pdf

        self.dinputs = dvalues * gelu_derivative


class Loss:
    def calculate(self,output,y):

        sample_losses = self.forward(output,y)
        data_loss = np.mean(sample_losses)
        return data_loss

class CCE(Loss):

    def forward(self,y_pred,y_true):
        sample = len(y_pred)
        y_pred_clipped = np.clip(y_pred,1e-7,1-1e-7)

        if len(y_true.shape)==1:
            correct_confidence = y_pred_clipped[range(sample),y_true]
        elif len(y_true.shape)==2:
            correct_confidence = np.sum(y_pred_clipped * y_true,axis=1)

        self.negative_log_likelihood = -np.log(correct_confidence)
        return self.negative_log_likelihood

    def backward(self,dvalues,y_true):

        sample = len(dvalues)
        labels = len(dvalues[0])

        if len(y_true.shape)==1:
            y_true = np.eye(labels)[y_true]

        self.dinputs = -y_true/dvalues
        self.dinputs = self.dinputs/sample


class Softmax_CCE:
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = CCE()

    def forward(self,inputs,y_true):
        self.activation.forward(inputs)
        self.outputs = self.activation.outputs

        return self.loss.calculate(self.outputs,y_true)

    def backward(self,dvalues,y_true):
        samples = len(dvalues)

        if len(y_true.shape)==2:
            y_true = np.argmax(y_true,axis=1)

        self.dinputs = dvalues.copy()

        self.dinputs[range(samples),y_true]-=1

        self.dinputs /= samples    





class Optimizer_SGD:

    def __init__(self,lr=0.01,decay=0.0):
        self.learning_rate = lr
        self.curr_lr = self.learning_rate
        self.decay = decay
        self.iter = 0
    def pre_update_params(self):
        if self.decay:
            self.curr_lr = (self.learning_rate/(1+self.decay * self.iter))
    def update_params(self,layer):
        layer.weights -= (self.curr_lr * layer.dweights)
        layer.bias -= (self.curr_lr * layer.dbias)

    def post_update_params(self):
        self.iter +=1


def load_mnist_images(filename):

    with open(filename,'rb') as f:
        f.read(16)
        images = np.frombuffer(f.read(),dtype=np.uint8)
    images = images.reshape(-1,28,28)

    return images

def load_mnist_labels(filename):

    with open(filename,'rb') as f:
        f.read(8)
        labels=np.frombuffer(f.read(),dtype=np.uint8)

    return labels

train_images = load_mnist_images(
    "data/mnist-numbers/train-images.idx3-ubyte"
)

train_labels = load_mnist_labels(
    "data/mnist-numbers/train-labels.idx1-ubyte"
)

test_images = load_mnist_images(
    "data/mnist-numbers/t10k-images.idx3-ubyte"
)

test_labels = load_mnist_labels(
    "data/mnist-numbers/t10k-labels.idx1-ubyte"
)

# print(train_images.shape)
# print(train_labels.shape)

# print(test_images.shape)
# print(test_labels.shape)
X= train_images.reshape(-1, 784).astype(np.float32)/255.0
y = train_labels
print(train_images.shape)
print(train_labels.shape)

batch_size = 128

dense1 = Add_layer(784,128)
dense2 = Add_layer(128,128)
dense3 = Add_layer(128,128)
dense4 = Add_layer(128,10)

activation1 = Activation_GeLu()
activation2 = Activation_GeLu()
activation3 = Activation_GeLu()

loss_activation = Softmax_CCE()

optimizer = Optimizer_SGD(lr=0.1,decay=1e-4)

prev_loss = 0
epochs=20
for epoch in range(epochs+1):
    epoch_loss = 0
    epoch_accuracy = 0
    steps = 0

    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    for start in range(0,len(X),batch_size):

        end = start + batch_size
        batch_X = X[start:end]
        batch_y = y[start:end]

        dense1.forward(batch_X)
        activation1.forward(dense1.outputs)

        dense2.forward(activation1.outputs)
        activation2.forward(dense2.outputs)

        dense3.forward(activation2.outputs)
        activation3.forward(dense3.outputs)

        dense4.forward(activation3.outputs)
        loss = loss_activation.forward(dense4.outputs,batch_y)

        predictions = np.argmax(loss_activation.outputs,axis=1)
        accuracy = np.mean(predictions == batch_y)

        epoch_loss += loss
        epoch_accuracy += accuracy
        steps += 1
        
        # print(f'Loss->{loss}  |  loss_diff->{loss-prev_loss}  |  accuracy->{accuracy}')
        # prev_loss = loss

        # backward

        loss_activation.backward(loss_activation.outputs,batch_y)

        dense4.backward(loss_activation.dinputs)
        activation3.backward(dense4.dinputs)
        dense3.backward(activation3.dinputs)
        activation2.backward(dense3.dinputs)
        dense2.backward(activation2.dinputs)
        activation1.backward(dense2.dinputs)
        dense1.backward(activation1.dinputs)

        optimizer.pre_update_params()
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.update_params(dense3)
        optimizer.update_params(dense4)
        optimizer.post_update_params()
    if epoch % 5 ==0:
        print(f"Epoch {epoch + 1}, "f"Loss: {epoch_loss / steps:.4f}, "f"Accuracy: {epoch_accuracy / steps:.4f}")


(60000, 28, 28)
(60000,)
Epoch 1, Loss: 0.4050, Accuracy: 0.8796
Epoch 6, Loss: 0.0718, Accuracy: 0.9781
Epoch 11, Loss: 0.0378, Accuracy: 0.9884
Epoch 16, Loss: 0.0224, Accuracy: 0.9940


In [13]:
X_test = test_images.reshape(-1,784).astype(np.float32)/255.0
y_test = test_labels

dense1.forward(X_test)
activation1.forward(dense1.outputs)

dense2.forward(activation1.outputs)
activation2.forward(dense2.outputs)

dense3.forward(activation2.outputs)
activation3.forward(dense3.outputs)

dense4.forward(activation3.outputs)

loss = loss_activation.forward(dense4.outputs, y_test)

predictions = np.argmax(loss_activation.outputs, axis=1)

accuracy = np.mean(predictions == y_test)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.0694
Test Accuracy: 0.9805
